## Package Install

In [ ]:
!pip install transformers


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load model and tokenizer
model_name = "valhalla/t5-small-qg-hl"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
!pip install pdfplumber

In [ ]:
import pdfplumber
import re

##Extract PDF Info

In [ ]:
#Extracting from PDF
def extract_text_from_pdf(pdf_path, output_path):
    all_text = []
    with pdfplumber.open(pdf_path) as pdf: #Open PDF
        for page in pdf.pages:
            text = page.extract_text()\
            cleaned_text = text.replace("-\n", "") #Account for line cutoffs by removing hyphens followed by newline
            all_text.append(cleaned_text)

    #Combine all text into one string
    full_text = "\n".join(all_text)

    #Save the text to a .txt file
    with open(output_path, 'w', encoding='utf-8') as file:
        file.write(full_text)

    print(f"Text successfully extracted and saved to {output_path}")


#Save Textbook to .txt
file_path = '/content/python_essentials_for_dummies.txt'
pdf_path = "/content/Stef Maruch, Aahz Maruch - Python For Dummies (2006, For Dummies) - libgen.li.pdf"
text = extract_text_from_pdf(pdf_path, file_path)

'\ndef extract_text_from_pdf(pdf_path, output_path):\n    all_text = []\n    with pdfplumber.open(pdf_path) as pdf:\n        for page in pdf.pages:\n            text = page.extract_text()\n            # Remove hyphenation at the end of lines (e.g., "com-\nputer" -> "computer")\n            cleaned_text = text.replace("-\n", "")  # Removes hyphens followed by a newline\n            all_text.append(cleaned_text)\n\n    # Combine all text into one string\n    full_text = "\n".join(all_text)\n\n    # Save the extracted text to a .txt file\n    with open(output_path, \'w\', encoding=\'utf-8\') as file:\n        file.write(full_text)\n\n    print(f"Text successfully extracted and saved to {output_path}")\n\n\n# Example usage\nfile_path = \'/content/python_essentials_for_dummies.txt\'\npdf_path = "/content/Stef Maruch, Aahz Maruch - Python For Dummies (2006, For Dummies) - libgen.li.pdf"\ntext = extract_text_from_pdf(pdf_path, file_path)\n'

## Chunk Text Function

In [ ]:
#Extracts words from saved txt file containing textbook content
file_path = '/content/python_essentials_for_dummies (2).txt'

#Read the content of the file into text variable
with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

In [ ]:
import re

def chunk_text(text, max_words=250):
    #Split the text into sentences using regex
    sentences = re.split(r'(?<=[.!?]) +', text)  #Split on sentence-ending punctuation

    chunks = []  #Store the final chunks
    current_chunk = []  #Store sentences for the current chunk
    current_word_count = 0  #Track number of words in the current chunk

    for sentence in sentences:
        word_count = len(sentence.split())

        #If adding this sentence exceeds the limit, finalize the current chunk
        if current_word_count + word_count > max_words:
            chunks.append(" ".join(current_chunk))  #Combine sentences into one chunk
            current_chunk = []  #Reset for the next chunk
            current_word_count = 0  #Reset the word count

        #Add the sentence to the current chunk
        current_chunk.append(sentence)
        current_word_count += word_count

    #Add the last chunk if there are leftover sentences
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

#Chunk textbook

chunks = list(chunk_text(text, max_words=150))

#Functions to use T5 to generate questions from each chunk and save questions to a .txt file

'''
def generate_questions_from_chunk(chunk):
    questions = []
    input_text = f"highlight: {chunk}" #Put chunk to make question
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True) #Tokenize input chunk
    outputs = model.generate(inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True) #Generate question from chunk
    question = tokenizer.decode(outputs[0], skip_special_tokens=True) #Decode question from token form
    questions.append(question)
    return questions

#Write questions to .txt file
def save_questions_to_file(questions, output_path="questions.txt"):
    with open(output_path, "w") as file:
        for question in questions:
            file.write(f"{question}\n")


all_questions = []
for chunk in chunks:
   all_questions.extend(generate_questions_from_chunk(chunk))

save_questions_to_file(all_questions)
'''

'\ndef generate_questions_from_chunk(chunk):\n    questions = []\n    input_text = f"highlight: {chunk}"\n    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)\n    outputs = model.generate(inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)\n    question = tokenizer.decode(outputs[0], skip_special_tokens=True)\n    questions.append(question)\n    return questions\n\ndef save_questions_to_file(questions, output_path="questions.txt"):\n    with open(output_path, "w") as file:\n        for question in questions:\n            file.write(f"{question}\n")\n\n\nall_questions = []\nfor chunk in chunks:\n   all_questions.extend(generate_questions_from_chunk(chunk))\n\nsave_questions_to_file(all_questions)\n'

#Extract .txt results of pdf and questions from google colab

In [ ]:
#Run chunk text
def chunk_text(text, max_words=250):
    #Split the text into sentences using regex
    sentences = re.split(r'(?<=[.!?]) +', text)  #Split on sentence-ending punctuation

    chunks = []  #Store final chunks
    current_chunk = []  #Store sentences for the current chunk
    current_word_count = 0  #Track number of words in the current chunk

    for sentence in sentences:
        word_count = len(sentence.split())

        #If adding this sentence exceeds the limit, finalize the current chunk
        if current_word_count + word_count > max_words:
            chunks.append(" ".join(current_chunk))  # Combine sentences into one chunk
            current_chunk = []  # Reset for the next chunk
            current_word_count = 0  # Reset the word count

        # Add the sentence to the current chunk
        current_chunk.append(sentence)
        current_word_count += word_count

    # Add the last chunk if there are leftover sentences
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

#Read questions.txt into a list
#with open("questions.txt", "r") as f:
  #  questions = [line.strip() for line in f.readlines()]

#Read the book.txt file into a list of sentences or paragraphs
i = 0
with open("/content/python_essentials_for_dummies (2).txt", "r") as f, open("text_chunked.txt", "w") as chunk:
    text = chunk_text(f.read(), max_words=100)
    for paragraph in text:
        chunk.write(paragraph + "\n")
        print(f"Chunk {i+1}: {paragraph}")
        i += 1

'''
from google.colab import files
files.download("text_chunked.txt")
'''

#print("Questions:", questions[:5])  # Print the first 5 questions
#print("Text sample:", text[:2])  # Print the first 2 paragraphs




Streaming output truncated to the last 5000 lines.
Tip This is similar to how function and module namespaces work—Python searches the
function's local namespace first, and then the module's global namespace, and then
the main or built-in namespace.
This gives rise to the following behaviors:
Inheriting from a single superclass
The names and definitions in a subclass or its instance override the same names and
definitions in the superclass.
Inheriting from multiple superclasses
When a class has more than one superclass, Python searches first the instance, and then
the subclass, as usual.
Chunk 694: When Python starts searching the superclasses, it uses a method
called depth-first, left-to-right searching, which searches in the following order:
1. Leftmost superclass in the class statement
For example, in the class statement
TotallyBizarreGreeeting(WeirdGreeting, NameTag), searching would
begin with WeirdGreeting, the leftmost superclass named.
2. Subclasses of the leftmost class
3. Next

# Context creation using sparse retrieval

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import csv
from transformers import T5Tokenizer

#Initialize the T-5 tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

questionContextSparceRetrieval = []

#Fit a TF-IDF Vectorizer on the documents
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(text)

for question in questions:

  query_vector = vectorizer.transform([question])

  #Compute similarity scores
  similarity_scores = np.dot(query_vector, doc_vectors.T).toarray()[0]

  #Retrieve the most relevant document(s)
  top_k_indices = np.argsort(similarity_scores)[::-1]  # Sort by descending score
  top_k_documents = [text[i] for i in top_k_indices[:2]]  # Top 2 results

  #Combine the top-k documents
  combined_context = " ".join(top_k_documents)

  #Tokenize the combined context into numerical IDs
  tokenized_context = tokenizer.encode(combined_context, truncation=True, max_length=512)

  #Append question and numerical context as a dictionary
  questionContextSparceRetrieval.append({
      "question": question,
      "context": tokenized_context
  })



#Convert to DataFrame
df = pd.DataFrame(questionContextSparceRetrieval)

#Save to CSV
df.to_csv("questions_with_sparse_context.csv", index=False, encoding="utf-8")

print("Data saved to questions_with_sparse_contextTokenized.csv")


Data saved to questions_with_sparse_contextTokenized.csv


# Context creation using Dense Retrieval

In [ ]:
!pip install -U sentence-transformers

In [ ]:
# !pip install --upgrade sympy
from sentence_transformers import SentenceTransformer, util
import numpy as np
import csv
import pandas as pd

questionContextDenseRetrieval = []
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(text, convert_to_tensor=True)


for question in questions:
  query_embedding = model.encode(question, convert_to_tensor=True)

  #Compute cosine similarity scores
  similarity_scores = util.cos_sim(query_embedding, doc_embeddings)[0]

  #Retrieve the most relevant document(s)
  top_k_indices = similarity_scores.argsort(descending=True)[:2]  # Top 2 results
  top_k_documents = [text[i] for i in top_k_indices]

   #Combine the top-k documents
  combined_context = " ".join(top_k_documents)

  #Tokenize the combined context into numerical IDs
  tokenized_context = tokenizer.encode(combined_context, truncation=True, max_length=512)

  #Append question and numerical context as a dictionary
  questionContextDenseRetrieval.append({
      "question": question,
      "context": tokenized_context
  })



#Convert to DataFrame
df = pd.DataFrame(questionContextDenseRetrieval)

#Saveto CSV
df.to_csv("questions_with_dense_context.csv", index=False, encoding="utf-8")

print("Data saved to questions_with_dense_context.csv")


KeyboardInterrupt: 

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import csv
import pandas as pd

# Ensemble Retrieval

In [ ]:
from sklearn.preprocessing import MinMaxScaler

#Normalize scores to [0, 1] using MinMaxScaler
def normalize(scores):
    scaler = MinMaxScaler()
    scores = scores.reshape(-1, 1)  #Reshape for scaler
    return scaler.fit_transform(scores).flatten()  #Flatten to 1D array

#Ensemble retrieval
def ensemble_retrieval(questions, text, alpha=0.5):
    results = []
    #Dense model
    dense_model = SentenceTransformer('all-MiniLM-L6-v2')
    doc_embeddings = dense_model.encode(text, convert_to_tensor=True)

    #Sparse model
    vectorizer = TfidfVectorizer()
    doc_vectors = vectorizer.fit_transform(text)

    for question in questions:
        #Dense retrieval
        query_embedding = dense_model.encode(question, convert_to_tensor=True)
        dense_similarity_scores = util.cos_sim(query_embedding, doc_embeddings)[0].cpu().numpy()

        #Sparse retrieval
        sparse_query_vector = vectorizer.transform([question])
        sparse_similarity_scores = np.dot(sparse_query_vector, doc_vectors.T).toarray()[0]

        #Normalize scores
        dense_scores_norm = normalize(dense_similarity_scores)
        sparse_scores_norm = normalize(sparse_similarity_scores)

        #Combine scores using weighted average
        combined_scores = alpha * sparse_scores_norm + (1 - alpha) * dense_scores_norm

        #Retrieve top-k documents
        top_k_indices = np.argsort(combined_scores)[::-1][:2]  # Top 2 results
        top_k_documents = [text[i] for i in top_k_indices]

        #Tokenize combined context
        combined_context = " ".join(top_k_documents)
        tokenized_context = tokenizer.encode(combined_context, truncation=True, max_length=512)

        #Append results
        results.append({
            "question": question,
            "context": tokenized_context
        })

    return results

#Perform ensemble retrieval
ensemble_results = ensemble_retrieval(questions, text, alpha=0.5)

#Convert to DataFrame and save to .csv
ensemble_df = pd.DataFrame(ensemble_results)
ensemble_df.to_csv("questions_with_ensemble_context.csv", index=False, encoding="utf-8")

print("Data saved to questions_with_ensemble_context.csv")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Data saved to questions_with_ensemble_context.csv


#Decodes and Generates Answers for CSV

In [ ]:
!pip install openai --upgrade
import openai
print(openai.__version__)



1.57.0


In [ ]:
import pandas as pd
import os
from google.colab import userdata
from transformers import T5Tokenizer
import openai
from openai import OpenAI

#Secret key for OpenAI
secretkey = userdata.get('') # insert user created secret key
client = OpenAI(api_key=secretkey)

#Initialize the T5 tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def decode_t5_vector(vector):
    try:
        #If the vector is a serialized string, make it into a list
        if isinstance(vector, str):
            token_ids = eval(vector)  #Convert string representation of a list to a Python list
        elif isinstance(vector, list):
            token_ids = vector  #Directly use the list
        else:
            raise ValueError("Input vector must be a list or a string representation of a list.")

        #Decode the token IDs back into a string
        decoded_text = tokenizer.decode(token_ids, skip_special_tokens=True)
        return decoded_text
    except Exception as e:
        return f"Error decoding vector: {e}"

def get_answer_from_gpt(question, context):
  #Gives same prompt for all questions, asks Chat GPT 3.5 Turbo to only use given context found from IR
    try:
        messages = [
            {"role": "system", "content": '''You are a Large Language Model that focuses solely on QA prompting. You will be prompted under the format of "Context: {context here} Question: {question here}. You must answer the provided question only using the context provided to you. Do not ever rely on any external knowledge or assumptions outside of the given context provided to you.'''},
            {"role": "user", "content": f"Context: {context}\nQuestion: {question}"}
        ]

        #Make the API call to OpenAI
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            max_tokens=75,
            temperature=0.7
        )

        #Extract and return the answer
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error generating answer: {e}"

def generate_answers(file_path, output_path):
    #Read the .csv file
    df = pd.read_csv(file_path)

    #Ensure the necessary columns are present
    required_columns = {'question', 'context'}
    if not required_columns.issubset(df.columns):
        print(f"Error: The CSV must contain the following columns: {required_columns}")
        return

    #Add an 'answer' column if it doesn't exist
    if 'answer' not in df.columns:
        df['answer'] = None

    #Iterate through each row to decode context and generate answers
    for index, row in df.iterrows():
        print(f"\n--- Processing Row {index} ---")
        print(f"Question: {row['question']}\n")

        #Decode the context
        decoded_context = decode_t5_vector(row['context'])
        print(f"Decoded Context: {decoded_context}\n")

        #Generate an answer using GPT
        gpt_answer = get_answer_from_gpt(row['question'], decoded_context)
        print(f"Generated Answer: {gpt_answer}\n")

        #Update the DataFrame with the GPT answer
        df.at[index, 'answer'] = gpt_answer

    #Save the updated CSV file
    df.to_csv(output_path, index=False)
    print(f"Answers appended. File saved at: {output_path}")

#Answer generation, plug in different files to text answer generation using various IR techniques
generate_answers('/content/questions_with_ensemble_context.csv', '/content/answers_ensemble.csv')



--- Processing Row 0 ---
Question: ﻿What is the maximum precision of decimal numbers in the decimal module?

Decoded Context: To see the parameters, type the following: >>> import decimal >>> decimal.getcontext() Context(prec=28, rounding=ROUND_HALF_EVEN, Emin=-999999999, Emax=999999999, capitals=1, flags=[], traps=[DivisionByZero, InvalidOperation, Overflow]) One of the parameters is the maximum precision of decimal numbers (how many digits to the right of the decimal point you see). To change the precision parameter, type something like this: >>> decimal.getcontext().prec = 7 Tip If you want to change the precision of a specific number without changing the precision parameter for all the numbers, use the quantize() method. quantize() rounds the number to a specific number of decimal places. Its results also match calculations done by hand (math educators take note!). 106 The decimal module gives more accurate results than binary floating point math for these types of calculations: C

##Annotate Sentences in CSV

In [ ]:
import pandas as pd
from transformers import T5Tokenizer
import re
import getch  #Allows single-key input without Enter

#Initialize the tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def decode_t5_vector(vector):
  #Decoding function used again
    try:
        if isinstance(vector, str):
            token_ids = eval(vector)
        elif isinstance(vector, list):
            token_ids = vector
        else:
            raise ValueError("Input vector must be a list or a string representation of a list.")
        return tokenizer.decode(token_ids, skip_special_tokens=True)
    except Exception as e:
        return f"Error decoding vector: {e}"

def sent_tokenize(text):
    #Splits the text into sentences using regex.
    sentences = re.split(r'(?<=[.!?]) +', text)
    return sentences

#Sentence annotation function, actually used on annotater's personal device to get 3 separate .csv files which were then combined
def annotate_hallucinations(file_path, output_path):
    #Read the .csv file
    df = pd.read_csv(file_path)

    #Ensure the necessary columns are present
    required_columns = {'question', 'context', 'answer'}
    if not required_columns.issubset(df.columns):
        print(f"Error: The CSV must contain the following columns: {required_columns}")
        return

    #Add new columns if they don't exist
    if 'Answer_sent_tokenized' not in df.columns:
        df['Answer_sent_tokenized'] = None
    if 'Sentence_labels' not in df.columns:
        df['Sentence_labels'] = None
    if 'Answerable' not in df.columns:
        df['Answerable'] = None
    if 'Does_not_answer' not in df.columns:
        df['Does_not_answer'] = None

    #Iterate through each row and process annotations
    for index, row in df.iterrows():
        print("\n--- Annotating Row {} ---".format(index))
        print(f"Question: {row['question']}\n")
        print(f"Decoded Context: {decode_t5_vector(row['context'])}\n")

        #Labels to indicate if the LLM would be able to answer the question based on the given context
        while True:
            print("Is the context relevant to the question? (Press 1 for True, 2 for False): ")
            answerable = getch.getch()  #Single key press
            if answerable in {'1', '2'}:
                df.at[index, 'Answerable'] = True if answerable == '1' else False
                break


        #Tokenize the answer into sentences
        sentences = sent_tokenize(row['answer'])
        labeled_sentences = {}
        tokenized_answers = []

        for i, sentence in enumerate(sentences, start=1):
            token_key = f"R{i}"
            print(f"{token_key}: {sentence}")
            tokenized_answers.append(f"{token_key}: {sentence}")

            while True:
                print("Enter 1 for 'Supported', 2 for 'Not Supported', or 3 for 'Neither': ")
                label = getch.getch()  #Single key press, more efficient annotation
                if label in {'1', '2', '3'}:
                    labeled_sentences[token_key] = {
                        '1': 'Supported',
                        '2': 'Not Supported',
                        '3': 'Neither'
                    }[label]
                    break

        while True:
              print("Did the LLM Answer? Press 1 for yes, 2 for no: ")
              does_not_answer = getch.getch()  #Single key press
              if does_not_answer in {'1', '0'}:
                  df.at[index, 'Does_not_answer'] = True if does_not_answer == '2' else False
                  break

        #Update the DataFrame with the tokenized answers and labels
        df.at[index, 'Answer_sent_tokenized'] = str(tokenized_answers)
        df.at[index, 'Sentence_labels'] = str(labeled_sentences)

    #Save the annotated CSV file
    df.to_csv(output_path, index=False)
    print(f"Annotation complete. File saved at: {output_path}")

#Run the annotation process
annotate_hallucinations('SparseAnswers.csv', 'Annotations.csv')



--- Annotating Row 0 ---
Question: s.upper() Returns string s with all letters converted to uppercase?

Decoded Context: Returns –1 if string x not found. s.rindex() Same as s.rfind but raises an error if the substring isn’t found. s.rstrip() Returns string s with any trailing whitespace removed. s.strip() Returns string s with leading and trailing whitespace removed. s.swapcase() Returns string s with uppercase letters converted to lowercase and lowercase letters converted to uppercase. s.title() Returns string s with the first letter of every word capitalized and all other letters lowercase. s.upper() Returns string s with all letters converted to uppercase. Don’t bother trying to memorize or even make sense of every string method. Returns –1 if none found. s.index(x, [y, Similar to find but raises a “substring not found” error if z]) string x can’t be found in string y. s.isalpha() Returns True if s is at least one character long and contains only letters (A–Z or a–z) and printable

KeyError: 'answer'

In [ ]:
import pandas as pd

#Tester function for context decoding function
def decode_and_print_context_from_csv(csv_file_path):
    #Read the .csv file
    df = pd.read_csv(csv_file_path)

    #Ensure the necessary column is present
    if 'context' not in df.columns:
        print(f"Error: The CSV must contain a 'context' column.")
        return

    #Iterate through each row and decode the context
    for index, row in df.iterrows():
        print(f"\n--- Decoding Context for Row {index} ---")
        print(f"Question: {row['question']}")

        #Decode the tokenized context
        decoded_context = decode_t5_vector(row['context'])

        #Print the decoded context
        print(f"Decoded Context: {decoded_context}")

#Tests decoding on encoded context
decode_and_print_context_from_csv("/content/questions_with_ensemble_context.csv")


##Detecting Hallucination with Keywords

In [ ]:
!pip install keybert

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import csv
from transformers import T5Tokenizer

In [ ]:
import pandas as pd
from keybert import KeyBERT
from sklearn.metrics.pairwise import cosine_similarity
from transformers import T5Tokenizer
from sentence_transformers import SentenceTransformer

#Initialize models
kw_model = KeyBERT()  #Load KeyBERT for keyword extraction
tokenizer = T5Tokenizer.from_pretrained("t5-small")  #Initialize T5 tokenizer for decoding
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  #Load SentenceTransformer for semantic similarity

#Function to decode T5 tokenized context
def decode_t5_vector(vector):
  #Reused function from previous need to decode context
    try:
        if isinstance(vector, str):
            token_ids = eval(vector)
        elif isinstance(vector, list):
            token_ids = vector
        else:
            raise ValueError("Input vector must be a list or a string representation of a list.")

        return tokenizer.decode(token_ids, skip_special_tokens=True)
    except Exception as e:
        return f"Error decoding vector: {e}"

#Extract top 5 keywords using KeyBERT
def extract_keywords(text, kw_model, top_n=5):
    try:
        return [kw[0] for kw in kw_model.extract_keywords(text, top_n=top_n)]
    except Exception as e:
        return []

#Compute keyword similarity for answer and context
def compute_keyword_similarity(answer, context, kw_model, embedding_model, top_n=5):
    #Extract keywords
    answer_keywords = extract_keywords(answer, kw_model, top_n=top_n)
    context_keywords = extract_keywords(context, kw_model, top_n=top_n)

    #Combine keywords into single strings for embedding
    answer_keywords_str = ' '.join(answer_keywords)
    context_keywords_str = ' '.join(context_keywords)

    #Compute embeddings
    answer_embedding = embedding_model.encode([answer_keywords_str])
    context_embedding = embedding_model.encode([context_keywords_str])

    #Calculate similarity
    return cosine_similarity(answer_embedding, context_embedding)[0][0]

#Process the .csv file
def process_csv_keywords(file_path, output_path, kw_model, embedding_model, threshold=0.46, top_n=5):
    df = pd.read_csv(file_path)

    #Initialize lists to store results
    hallucination_detected_list = []
    similarity_score_list = []
    decoded_context_list = []

    for index, row in df.iterrows():
        question = row['question']
        context = row['context']
        answer = row['answer']

        #Decode the context
        context_vector = eval(context)
        decoded_context = decode_t5_vector(context_vector)
        decoded_context_list.append(decoded_context)

        #Compute keyword similarity score
        keyword_similarity_score = compute_keyword_similarity(answer, decoded_context, kw_model, embedding_model, top_n=top_n)
        similarity_score_list.append(keyword_similarity_score)

        #Detect hallucination
        hallucination_detected = True if keyword_similarity_score < threshold else False
        hallucination_detected_list.append(hallucination_detected)

        #Print progress
        print(f"Row {index+1}:")
        print(f"Question: {question}")
        print(f"Decoded Context: {decoded_context}")
        print(f"Answer: {answer}")
        print(f"Keyword Similarity Score: {keyword_similarity_score:.4f}")
        print(f"Hallucination Detected: {hallucination_detected}\n")

    #Add results to DataFrame
    df['Keyword Similarity Score'] = similarity_score_list
    df['Hallucination Detected'] = hallucination_detected_list

    #Save to output .csv
    df.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")

if __name__ == "__main__":
    input_csv_path = '/content/SparseHallucinationsCombined.csv'  #Path to input .csv
    output_csv_path = '/content/SparseHallucinationsDetectedKeywords.csv'  #Path to output .csv

    process_csv_keywords(input_csv_path, output_csv_path, kw_model, embedding_model, threshold=0.46, top_n=5)


Row 1:
Question: How do you import sibling modules in a package?
Decoded Context: If you don't, someone reading your import statement has no way of knowing whether it imports another module from the package or a top-level package that's part of the Python distribution. TECHNICAL Although it's possible to use relative imports in Python 2.4 and STUFF earlier, the rules are too complicated to document in this book. Absolute importing, new in Python 2.5, addresses this problem. There's no ambiguity about which module is being imported because Python searches for imported modules or packages on sys.path only. To import sibling modules in a package, use relative imports (see the following section). If you write the swirl.py module in the package hierarchy (described in "Example package structure," in this chapter), you import other modules in the package as follows: • This statement imports swirl.py's sibling module, speckle.py: • from. import speckle • This statement imports the gifread mod

In [ ]:
!pip install transformers
!pip install sentence-transformers
!pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


##Embedding Similairites

In [ ]:
import pandas as pd
from transformers import T5Tokenizer
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import CountVectorizer
import logging

#Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

#Initialize the tokenizers and models
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
vectorizer = CountVectorizer(stop_words="english")

#Function to decode T5 tokenized context
def decode_t5_vector(vector):
    try:
        if isinstance(vector, str):
            token_ids = eval(vector)
        elif isinstance(vector, list):
            token_ids = vector
        else:
            raise ValueError("Input vector must be a list or a string representation of a list.")

        return t5_tokenizer.decode(token_ids, skip_special_tokens=True)
    except Exception as e:
        return f"Error decoding vector: {e}"

#Function to detect hallucination
def detect_hallucination(row):
    try:
        #Decode the context
        decoded_context = decode_t5_vector(row["context"])
        combined_qa = f"Q: {row['question']} A: {row['answer']}"

        #Compute cosine similarity
        context_embedding = embedding_model.encode(decoded_context, convert_to_tensor=True)
        qa_embedding = embedding_model.encode(combined_qa, convert_to_tensor=True)
        similarity_tensor = util.cos_sim(context_embedding, qa_embedding)
        similarity = similarity_tensor.item()  #Find similarity scalar value

        #If similarity score below a certain
        is_similarity_low = similarity < 0.55
        hallucinated = is_similarity_low

        #Return both hallucination status and similarity score
        return similarity, hallucinated
    except Exception as e:
        logging.error(f"Error processing row: {e}")
        return None, None

#Function to process a .csv file containing a table with questions, contexts, and answers
def process_csv_embeddings(input_file, output_file, batch_size=1000):
    try:
        #Open input .csv in chunks
        chunk_iter = pd.read_csv(input_file, chunksize=batch_size)

        #Create output file with initial structure
        first_chunk = next(chunk_iter)
        first_chunk[["Similarity Score", "Hallucination Detected"]] = first_chunk.apply(
            lambda row: pd.Series(detect_hallucination(row)), axis=1
        )
        first_chunk.to_csv(output_file, index=False)

        #Process remaining chunks and append to the output file
        for chunk in chunk_iter:
            logging.info(f"Processing a new batch of size: {len(chunk)}")
            chunk[["Similarity Score", "Hallucination Detected"]] = chunk.apply(
                lambda row: pd.Series(detect_hallucination(row)), axis=1
            )
            chunk.to_csv(output_file, mode="a", header=False, index=False)

        logging.info(f"Processing completed. Results saved to {output_file}.")
    except Exception as e:
        logging.error(f"Error processing the CSV file: {e}")


#Run the script
if __name__ == "__main__":
    input_csv = "/content/SparseHallucinationsCombined.csv"
    output_csv = "/content/SparseHallucinationsDetectedEmbeddings.csv"

    #Process the .csv file
    procesprocess_csv_embeddingss_csv(input_csv, output_csv)


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#Function to calculate evaluation metrics for hallucination detection methods
def evaluate_hallucination_detection(file_path):
    #Load the .csv file
    df = pd.read_csv(file_path)

    #Extract the true and predicted labels
    true_labels = df['Hallucinated']  #True labels from the 'Hallucinated' column
    predicted_labels = df['Hallucination Detected']  #Predicted labels from the 'Hallucination Detected' column

    #Calculate evaluation metrics
    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels)
    recall = recall_score(true_labels, predicted_labels)
    f1 = f1_score(true_labels, predicted_labels)

    # Print the results
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

#Run the evaluation on all .csv files
if __name__ == "__main__":
    input_csv_path = '/content/SparseHallucinationsDetectedKeywords.csv'
    print("Sparse Hallucination Prediction Using Keywords")
    evaluate_hallucination_detection(input_csv_path)

    input_csv_path = '/content/SparseHallucinationsDetectedEmbeddings.csv'
    print("Sparse Hallucination Prediction Using Embeddings")
    evaluate_hallucination_detection(input_csv_path)


    input_csv_path = '/content/EnsembleHallucinationsDetectedKeywords.csv'
    print("Ensemble Hallucination Prediction Using Keywords")
    evaluate_hallucination_detection(input_csv_path)

    input_csv_path = '/content/EnsembleHallucinationsDetectedEmbeddings.csv'
    print("Ensemble Hallucination Prediction Using Embeddings")
    evaluate_hallucination_detection(input_csv_path)



Sparse Hallucination Prediction Using Keywords
Accuracy: 0.6878
Precision: 0.3571
Recall: 0.4630
F1 Score: 0.4032
Sparse Hallucination Prediction Using Embeddings
Accuracy: 0.7046
Precision: 0.3750
Recall: 0.4444
F1 Score: 0.4068
Ensemble Hallucination Prediction Using Keywords
Accuracy: 0.7637
Precision: 0.2250
Recall: 0.2647
F1 Score: 0.2432
Ensemble Hallucination Prediction Using Embeddings
Accuracy: 0.7932
Precision: 0.3077
Recall: 0.3529
F1 Score: 0.3288


# Results

In [ ]:
import pandas as pd
import ast  #For safely parsing the dictionary-like string in 'Sentence_labels'

#Function to count the number of 'Not Supported' in Sentence_labels
def count_sentence_labels(df):
    not_supported_count = 0
    supported_count = 0
    neither_count = 0

    for sentence_labels in df['Sentence_labels']:
        #Convert the string representation of the dictionary to a real dictionary
        labels_dict = ast.literal_eval(sentence_labels)

        #Count occurrences for each category
        not_supported_count += sum(1 for value in labels_dict.values() if value == 'Not Supported')
        supported_count += sum(1 for value in labels_dict.values() if value == 'Supported')
        neither_count += sum(1 for value in labels_dict.values() if value == 'Neither')

    return not_supported_count, supported_count, neither_count

#Function to count TRUE and FALSE in the 'Answerable' column
def count_answerable(df):
    true_count = df['Answerable'].sum()  # Assuming TRUE is represented as 1
    false_count = len(df) - true_count    # The rest are FALSE (assuming only TRUE/FALSE)
    return true_count, false_count

#Function to count TRUE and FALSE in the 'Does_not_answer' column
def count_does_not_answer(df):
    true_count = df['Does_not_answer'].sum()  #Assuming TRUE is represented as 1
    false_count = len(df) - true_count         # he rest are FALSE (assuming only TRUE/FALSE)
    return true_count, false_count

# Function to count TRUE and FALSE in the 'Does_not_answer' column
def count_Hallucinated(df):
    true_count = df['Hallucinated'].sum()  #Assuming TRUE is represented as 1
    false_count = len(df) - true_count         #The rest are FALSE (assuming only TRUE/FALSE)
    return true_count, false_count

def results(df):
    #Call the functions
    not_supported_count, supported_count, neither_count = count_sentence_labels(df)
    answerable_true, answerable_false = count_answerable(df)
    does_not_answer_true, does_not_answer_false = count_does_not_answer(df)
    hallucinated_true, hallucinated_false = count_Hallucinated(df)

    #Print the results
    print()
    print(f"Sentence Labels - Not Supported: {not_supported_count}, Supported: {supported_count}, Neither: {neither_count}")
    print(f"Answerable - TRUE: {answerable_true}, FALSE: {answerable_false}")
    print(f"Does_not_answer - TRUE: {does_not_answer_true}, FALSE: {does_not_answer_false}")
    print(f"Hallucinated - TRUE: {hallucinated_true}, FALSE: {hallucinated_false},  Hallucination Rate: {hallucinated_true / (hallucinated_true + hallucinated_false) * 100:.2f}%")
    print("\n")

df = pd.read_csv('/content/SparseHallucinationsCombined.csv')
print("Sparse Results")
results(df)

df = pd.read_csv('/content/EnsembleHallucinationsCombined.csv')
print("Ensemble Results")
results(df)


Sparse Results

Sentence Labels - Not Supported: 45, Supported: 275, Neither: 22
Answerable - TRUE: 201, FALSE: 36
Does_not_answer - TRUE: 7, FALSE: 230
Hallucinated - TRUE: 54, FALSE: 183,  Hallucination Rate: 22.78%


Ensemble Results

Sentence Labels - Not Supported: 42, Supported: 312, Neither: 13
Answerable - TRUE: 218, FALSE: 19
Does_not_answer - TRUE: 7, FALSE: 230
Hallucinated - TRUE: 34, FALSE: 203,  Hallucination Rate: 14.35%


